
# Public Demo Setup / Initialization Notebook

This notebook is **not** the public demo itself. It is the **private preparation notebook** you run once to:

1. point the repo at your Hugging Face model repo
2. export a tiny public MAESTRO demo subset from your private Drive/cache
3. upload the public checkpoint + demo assets to Hugging Face
4. sanity-check that the public demo layer can download and load the checkpoint

Use this notebook whenever you want to refresh the public checkpoint or update the curated sample set.


In [ ]:

#@title 0. Setup repo access and install helper dependencies
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  #@param {type:"string"}
REPO_BRANCH = "main"  #@param {type:"string"}
REPO_CLONE_DIR = "/content/repo"  #@param {type:"string"}

repo_dir = Path(REPO_CLONE_DIR)
if not repo_dir.exists():
    !git clone -q --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_CLONE_DIR"
else:
    print(f"Repo already present at {repo_dir}")

%cd $REPO_CLONE_DIR
!pip -q install -r requirements_demo.txt huggingface_hub


In [ ]:

#@title 1. Imports
import json
import os
import sys
from pathlib import Path

from huggingface_hub import notebook_login

if REPO_CLONE_DIR not in sys.path:
    sys.path.insert(0, REPO_CLONE_DIR)

from demo.export_demo_assets import export_demo_assets
from demo.upload_to_hf import upload_demo_package



## 2. Hugging Face authentication

Run the next cell and log in with a Hugging Face token that has permission to create or update your model repo.


In [ ]:

#@title 2. Log in to Hugging Face
notebook_login()


In [ ]:

#@title 3. Private paths and public repo settings (edit these)
HF_REPO_ID = "YOUR_USERNAME/piano-amt-demo"  #@param {type:"string"}
PUBLIC_PRIVATE = False  #@param {type:"boolean"}

CHECKPOINT_PATH = "/content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt"  #@param {type:"string"}
MAESTRO_ROOT = "/content/drive/MyDrive/piano_amt/maestro-v3.0.0"  #@param {type:"string"}
CACHE_DIR = "/content/drive/MyDrive/piano_amt/cache"  #@param {type:"string"}
OUTPUT_DEMO_ASSETS = str(Path(REPO_CLONE_DIR) / "demo_assets")  #@param {type:"string"}

# Add 2–5 short stems from your private MAESTRO test set.
DEMO_STEMS = [
    "PUT_AUDIO_STEM_1_HERE",
    "PUT_AUDIO_STEM_2_HERE",
]

checkpoint_path = Path(CHECKPOINT_PATH)
maestro_root = Path(MAESTRO_ROOT)
cache_dir = Path(CACHE_DIR)
output_demo_assets = Path(OUTPUT_DEMO_ASSETS)
manifest_path = Path(REPO_CLONE_DIR) / "demo" / "sample_manifest.json"

print("HF repo:", HF_REPO_ID)
print("Checkpoint:", checkpoint_path)
print("MAESTRO root:", maestro_root)
print("Cache dir:", cache_dir)
print("Demo assets dir:", output_demo_assets)
print("Manifest path:", manifest_path)
print("Demo stems:", DEMO_STEMS)



## 4. Export a tiny public MAESTRO demo subset

This reads your **private** MAESTRO CSV/cache and writes only a few public demo files into your repo:

- `demo_assets/sample_01.wav`
- `demo_assets/sample_01_labels.npz`
- ...
- `demo/sample_manifest.json`


In [ ]:

#@title 4. Export demo assets from your private dataset/cache
export_demo_assets(
    maestro_root=maestro_root,
    cache_dir=cache_dir,
    output_dir=output_demo_assets,
    names=DEMO_STEMS,
)

print("\nManifest contents:\n")
print((manifest_path).read_text())



## 5. Upload checkpoint + demo assets to Hugging Face

This creates or updates the public model repo and uploads:

- `checkpoints/best.pt`
- `demo_assets/*`
- `demo/sample_manifest.json`


In [ ]:

#@title 5. Upload the public demo package to Hugging Face
upload_demo_package(
    repo_id=HF_REPO_ID,
    checkpoint=checkpoint_path,
    demo_assets=output_demo_assets,
    manifest=manifest_path,
    private=PUBLIC_PRIVATE,
)



## 6. Point the repo demo layer at the public HF repo

You can either:

1. edit `demo/demo_config.py` permanently, or
2. set environment variables inside the public notebook.

The public notebook generated for this repo uses environment variables in a config cell, so you usually do **not** need to hard-edit the Python file each time.


In [ ]:

#@title 6. Optional sanity check: test public checkpoint download/load
os.environ["AMT_DEMO_HF_REPO_ID"] = HF_REPO_ID
os.environ["AMT_DEMO_HF_CHECKPOINT_FILENAME"] = "checkpoints/best.pt"
os.environ["AMT_DEMO_REPO_ROOT"] = REPO_CLONE_DIR

from demo.checkpoint import load_demo_model, format_checkpoint_summary

model, ckpt, ckpt_path = load_demo_model("cpu")
print(format_checkpoint_summary(model, ckpt, ckpt_path))



## 7. What to do next

1. Commit and push the updated repo files:
   - `demo/`
   - `demo_assets/`
   - `requirements_demo.txt`
   - `notebooks/`
2. Open the public notebook from GitHub in Colab.
3. Edit the top config cell once with:
   - your GitHub repo URL
   - your GitHub branch
   - your Hugging Face repo ID
4. Test the full public flow in a fresh Colab runtime.
